# 🩺 Apresentação Final - Assistente Médico Multiagentes
**Tech Challenge - Fase 3 | Pós-Graduação IA para Devs - FIAP**
**Autor:** Marco Antonio Teixeira

Este notebook demonstra a integração final do projeto. Ele clona o repositório atualizado, carrega o modelo fine-tunado (QLoRA) do Google Drive e executa o orquestrador LangGraph com os 4 agentes especialistas e o Guardrail de segurança.

In [ ]:
# 1. Clonar o repositório atualizado do GitHub
!git clone https://github.com/MAntonioST/assistente-medico-llm.git
%cd assistente-medico-llm

# 2. Instalar dependências do projeto e pacotes para LLM/GPU
!pip install -r requirements.txt
!pip install -q peft bitsandbytes accelerate langchain-huggingface

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ ATENÇÃO: Ajuste este caminho para a pasta exata onde você salvou 
# o seu modelo treinado (adaptador QLoRA) no seu Google Drive!
ADAPTER_PATH = "/content/drive/MyDrive/FIAP/TechChallenge3/modelo_finetuned" 

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from peft import PeftModel

print("Carregando modelo base e adaptador...")
model_id = "HuggingFaceH4/zephyr-7b-beta"

# Carrega Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Carrega Modelo Base em 4-bits para economizar memória da GPU
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)

# Aplica o adaptador treinado (Fine-Tuning)
try:
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    print("✅ Adaptador QLoRA carregado com sucesso!")
except Exception as e:
    print(f"⚠️ Aviso: Não foi possível carregar o adaptador ({e}). Usando modelo base.")
    model = base_model

# Cria o pipeline do LangChain
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    repetition_penalty=1.1
)

llm_finetuned = HuggingFacePipeline(pipeline=pipe)
print("✅ LLM pronta para uso!")

In [ ]:
import sys
sys.path.append('/content/assistente-medico-llm')

from src.agent.workflow import AssistenteMedicoApp
from src.agent.exam_agent import AgenteExames
from src.agent.protocol_agent import AgenteProtocolos
from src.agent.pending_agent import PendingExamsAgent
from src.agent.alert_agent import AlertAgent
from src.guardrails.medical_guardrail import GuardrailMedico

print("Inicializando Agentes...")
agente_exames = AgenteExames()
# Aqui a mágica acontece: Injetamos a LLM real no agente de protocolos!
agente_protocolos = AgenteProtocolos(llm=llm_finetuned) 
agente_pendencias = PendingExamsAgent()
agente_alertas = AlertAgent()
guardrail = GuardrailMedico()

# Monta o orquestrador LangGraph
app = AssistenteMedicoApp(agente_exames, agente_protocolos, agente_pendencias, agente_alertas, guardrail)
print("✅ Sistema Multiagentes (LangGraph) pronto!")

### 🧪 Testando os Fluxos de Decisão (Roteamento Automático)

In [ ]:
# Teste 1: Agente de Protocolos (Usa RAG + LLM Treinada)
print("Pergunta: Qual o tratamento padrão para hipertensão?\n")
resposta = app.processar("Qual o tratamento padrão para hipertensão?")
print(f"🤖 Assistente:\n{resposta}")

In [ ]:
# Teste 2: Agente de Exames (Busca no JSON)
print("Pergunta: resultado do exame do robert johnson\n")
resposta = app.processar("resultado do exame do robert johnson")
print(f"🤖 Assistente:\n{resposta}")

In [ ]:
# Teste 3: Agente de Pendências
print("Pergunta: exames pendentes do robert johnson\n")
resposta = app.processar("exames pendentes do robert johnson")
print(f"🤖 Assistente:\n{resposta}")

In [ ]:
# Teste 4: Agente de Alertas
print("Pergunta: emitir alerta de emergência\n")
resposta = app.processar("emitir alerta de emergência")
print(f"🤖 Assistente:\n{resposta}")